# Chapter 1 · ETF 수익률 정상성·변동성

**연구질문:** ETF 가격과 로그수익률의 정상성은 어떻게 다른가? 수익률의 제곱에서 변동성 군집이 관찰되는가?

**학습 흐름:** 개념 → 자료 → 시각화 → 분해 → 정상성 검정 → 자기상관 → 해석.

> 이 노트북은 위에서 아래로 실행하면 끝까지 동작하도록 만들었습니다. 온라인 자료를 먼저 시도하고, 실패하면 **합성자료로 자동 대체**하며 `DATA_KIND`에 표시합니다.

## 1. 핵심 개념·수식

**약정상성(weak stationarity):** $E(y_t)=\mu$, $\mathrm{Var}(y_t)=\sigma^2$, $\mathrm{Cov}(y_t,y_{t-k})=\gamma(k)$는 $t$에 의존하지 않는다.

**고전적 분해(가법):** $y_t = T_t + S_t + R_t$. 계절성의 세기는 $F_S=\max\!\left(0,\,1-\dfrac{\mathrm{Var}(R_t)}{\mathrm{Var}(S_t+R_t)}\right)$로 요약한다(0에 가까우면 약함).

**단위근 검정:** ADF의 귀무가설은 *단위근 존재*, KPSS의 귀무가설은 *정상성*이다. 두 검정을 함께 읽는다.

| ADF (H0: 단위근) | KPSS (H0: 정상) | 읽는 법 |
|---|---|---|
| 기각 | 기각 못함 | 정상성과 부합 |
| 기각 못함 | 기각 | 단위근/비정상과 부합 → 차분 고려 |
| 둘 다 기각 | | 추세정상·구조변화·분산 불안정 가능 |
| 둘 다 기각 못함 | | 검정력 부족, 판단 보류 |

**자기상관:** $\rho(k)=\gamma(k)/\gamma(0)$. Ljung–Box $Q=n(n+2)\sum_{k=1}^{h}\dfrac{\hat\rho(k)^2}{n-k}$. 변동성 군집은 $r_t^2$의 자기상관(ARCH-LM)으로 점검한다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def garch_sim(n, seed, omega=0.02, alpha=0.08, beta=0.90):
    rng = np.random.default_rng(seed); r = np.zeros(n); s2 = omega / (1 - alpha - beta)
    for t in range(n):
        r[t] = np.sqrt(s2) * rng.standard_normal()
        s2 = omega + alpha * r[t] ** 2 + beta * s2
    return r + 0.04                                            # 일수익률(%) , 약간의 양의 평균

def load_finance():
    px, kind = None, "SYNTHETIC (NOT real data): GARCH(1,1) 수익률로 만든 가격"
    if USE_LIVE_DATA and ensure("yfinance"):
        try:
            import yfinance as yf
            raw = yf.download("SPY", start="2015-01-01", progress=False, auto_adjust=True)
            c = raw["Close"]
            if isinstance(c, pd.DataFrame):
                c = c.iloc[:, 0]
            c = c.dropna().astype(float)
            if len(c) > 500:
                px, kind = c, "REAL: SPY (Yahoo Finance, yfinance)"
        except Exception as e:
            print("온라인 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
    if px is None:
        r = garch_sim(2500, SEED)
        px = pd.Series(100 * np.exp(np.cumsum(r / 100)), index=pd.bdate_range("2015-01-01", periods=2500))
    px.index = pd.to_datetime(px.index)
    if getattr(px.index, "tz", None) is not None:
        px.index = px.index.tz_localize(None)
    return dict(y=px.rename("price"), kind=kind, m=5, freq="B", label="SPY 종가", unit="USD")

D = load_finance()
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
ret = (np.log(y).diff().dropna() * 100).rename("log_return_pct")      # 로그수익률(%)
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 시각화와 진단

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
y.plot(ax=axes[0], title=f"Price | {DATA_KIND}")
ret.plot(ax=axes[1], title="일별 로그수익률(%)")
plt.tight_layout(); plt.show()
print(ret.describe().round(4).to_string())
print("왜도:", round(float(stats.skew(ret)), 3), "| 초과첨도:", round(float(stats.kurtosis(ret)), 3),
      "(정규분포면 둘 다 0)")

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.seasonal import seasonal_decompose

def stationarity_row(s, label):
    x = pd.Series(s).dropna().to_numpy(dtype=float)
    adf_p = adfuller(x, autolag="AIC")[1]
    kpss_p = kpss(x, regression="c", nlags="auto")[1]
    verdict = ("정상성과 부합" if (adf_p < 0.05 and kpss_p >= 0.05) else
               "비정상 의심(차분 고려)" if (adf_p >= 0.05 and kpss_p < 0.05) else
               "두 검정 불일치/검정력 한계: 판단 보류")
    return {"series": label, "ADF_p": adf_p, "KPSS_p": kpss_p, "읽기": verdict}

In [ ]:
res = pd.DataFrame([stationarity_row(y, "Price"), stationarity_row(ret, "Log return(%)")]); show(res)
work = ret; Fs = np.nan

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
nl = int(min(40, len(work) // 4))
plot_acf(work, lags=nl, ax=ax[0], title="ACF"); plot_pacf(work, lags=nl, ax=ax[1], title="PACF", method="ywm")
plt.tight_layout(); plt.show()
lb = acorr_ljungbox(work, lags=[10], return_df=True)
print("Ljung-Box(10) p =", round(float(lb["lb_pvalue"].iloc[0]), 4), "-> 작으면 자기상관 존재")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
plot_acf(ret ** 2, lags=40, ax=ax[0], title="제곱수익률 ACF")
ret.abs().rolling(21).mean().plot(ax=ax[1], title="21일 평균 |수익률| (변동성 군집)")
plt.tight_layout(); plt.show()
arch_p = het_arch(ret, nlags=10)[1]
print("ARCH-LM(10) p =", round(float(arch_p), 4), "-> 작으면 조건부 이분산(변동성 군집)과 부합")

In [ ]:
print("=== 자동 요약(숫자 확인용, 해석은 직접 작성) ===")
print("데이터 종류:", DATA_KIND)
for r_ in res.to_dict("records"):
    print(f"- {r_['series']}: ADF p={r_['ADF_p']:.3f}, KPSS p={r_['KPSS_p']:.3f} -> {r_['읽기']}")
if not np.isnan(Fs): print(f"- 계절성 세기 F_S={Fs:.2f}")

## 3. 결과 해석 작성

1. `DATA_KIND`가 **REAL**인지 **SYNTHETIC**인지 먼저 적는다. 합성자료의 결과는 방법 시연이며 현상에 대한 증거가 아니다.
2. 수준과 차분(또는 수익률)에서 ADF·KPSS가 어떻게 다른지 설명한다. 두 검정이 어긋나면 구조변화·추세·표본기간을 논의한다.
3. 계절성 세기와 ACF/PACF가 이후 모형(Ch.2)의 후보 차수에 주는 시사점을 적는다.
4. 한계: 단일 표본, 검정력, 데이터 정의(재분석·집계·취소 제외 등).

## 4. 심화 과제
- 표본기간을 절반으로 나누어 결론이 유지되는지 확인한다.
- 차분 대신 로그변환 후 차분으로 바꿔 결과를 비교한다.
- **제출:** 그래프 3개 이상, 검정표, 해석 단락, 데이터 출처·한계.